# Malasanità in Italia — Preanalysis v2 (Euro-2013 proxy)
**Anno 2022 | Fonti: Ministero della Salute (A, C) + ISTAT (D)**

**Domanda:** Le regioni con meno personale sanitario hanno tassi di mortalità evitabile più alti?

Questa preanalysis usa il **Euro-2013 proxy**: 12 cause amenable/preventable dalla fonte D ISTAT.
Differenza rispetto a v1: la mortalità è filtrata per cause evitabili (non totale). Il risultato è un tasso grezzo 30+, non age-standardized.

*Cause incluse: Sepsi, Colon-retto, Polmone, Seno, Diabete, Ischemiche cuore, Cerebrovascolari, Ipertensive, Influenza/Polmonite, BPCO, Cirrosi, Cause esterne.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

cwd = Path.cwd().resolve()
project_root = next(
    (path for path in [cwd, *cwd.parents]
     if (path / 'sources').exists() and (path / 'compose').exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Non trovo la root di malasanita-struttura-mortalita')

repo_root = project_root.parents[2]
MART_PATH = repo_root / 'out' / 'data' / 'mart' / 'malasanita_a_strutture_asl' / '2022' / 'mart_compose_regioni.parquet'

df = pd.read_parquet(MART_PATH)
print(f'Regioni/PA caricate: {len(df)} | Anno: {df["anno"].iloc[0]}')
print(f'Join C ok: {df["join_c_ok"].sum()}/21 | Join D ok: {df["join_d_ok"].sum()}/21')
print(f'Colonna mortalita: decessi_evitabili_30plus_per_100k_pop_totale')

## Panoramica regionale 2022

Tabella ordinata per mortalità evitabile decrescente.

In [ ]:
cols_display = [
    'regione',
    'medici_mmg_per_100k',
    'infermieri_per_100k',
    'personale_osp_per_100k',
    'posti_letto_previsti_per_100k',
    'decessi_evitabili_30plus_per_100k_pop_totale',
]

tab = (
    df[cols_display]
    .sort_values('decessi_evitabili_30plus_per_100k_pop_totale', ascending=False)
    .reset_index(drop=True)
)
tab.columns = ['Regione', 'MMG/100k', 'Infermieri/100k', 'Pers.osp/100k', 'PL/100k', 'Evitabili30+/100k']
tab.index += 1
tab

## Numeri chiave

In [ ]:
mort = df['decessi_evitabili_30plus_per_100k_pop_totale']
mmg  = df['medici_mmg_per_100k']
inf  = df['infermieri_per_100k']

r_mort_max = df.loc[mort.idxmax(), 'regione']
r_mort_min = df.loc[mort.idxmin(), 'regione']
r_mmg_max  = df.loc[mmg.idxmax(),  'regione']
r_mmg_min  = df.loc[mmg.idxmin(),  'regione']

print("-- MORTALITA' EVITABILE 30+ per 100k (Euro-2013 proxy) --")
print(f'  Piu alta:  {r_mort_max:<30} {mort.max():.0f}')
print(f'  Piu bassa: {r_mort_min:<29} {mort.min():.0f}')
print(f'  Scarto max-min: {mort.max()-mort.min():.0f} decessi evitabili per 100k')
print(f'  Media nazionale: {mort.mean():.0f}')
print()
print('-- MMG per 100k --')
print(f'  Piu alto:  {r_mmg_max:<30} {mmg.max():.1f}')
print(f'  Piu basso: {r_mmg_min:<29} {mmg.min():.1f}')

## Scatter — MMG vs mortalità evitabile 30+ (Euro-2013)

Asse X: MMG per 100k residenti (fonte A) | Asse Y: decessi evitabili 30+ per 100k (Euro-2013 proxy)

**Atteso:** correlazione negativa — più MMG, meno mortalità evitabile.

In [ ]:
def sigla(nome):
    m = {
        'Valle': 'VDA', 'Bolzano': 'BZ', 'Trento': 'TN',
        'Friuli': 'FVG', 'Emilia': 'ER', 'Toscana': 'TOS',
        'Umbria': 'UMB', 'Liguria': 'LIG', 'Lombardia': 'LOM',
        'Piemonte': 'PIE', 'Veneto': 'VEN', 'Marche': 'MAR',
        'Lazio': 'LAZ', 'Abruzzo': 'ABR', 'Molise': 'MOL',
        'Campania': 'CAM', 'Puglia': 'PUG', 'Basilicata': 'BAS',
        'Calabria': 'CAL', 'Sicilia': 'SIC', 'Sardegna': 'SAR',
    }
    for k, v in m.items():
        if k in nome:
            return v
    return nome[:3].upper()


def scatter_plot(ax, x_col, y_col, x_label, title, df):
    x = df[x_col]
    y = df[y_col]
    ax.scatter(x, y, color='#2166ac', alpha=0.85, s=60, zorder=3)
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color='#d62728', linewidth=1.2,
            linestyle='--', alpha=0.7, label=f'Regressione (m={m:.0f})')
    r = np.corrcoef(x, y)[0, 1]
    ax.text(0.97, 0.97, f'r = {r:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='#555555')
    for _, row in df.iterrows():
        ax.annotate(sigla(row['regione']), (row[x_col], row[y_col]),
                    textcoords='offset points', xytext=(5, 3), fontsize=7.5, color='#333333')
    ax.set_xlabel(x_label, fontsize=9)
    ax.set_ylabel('Decessi evitabili 30+ per 100k (Euro-2013 proxy)', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, linestyle=':')


fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(ax, 'medici_mmg_per_100k', 'decessi_evitabili_30plus_per_100k_pop_totale',
             'Medici di base (MMG) per 100k residenti',
             'MMG per 100k vs Mortalita evitabile 30+ (Euro-2013) — Regioni 2022', df)
plt.tight_layout()
plt.show()

## Scatter — Infermieri vs mortalità evitabile 30+

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(ax, 'infermieri_per_100k', 'decessi_evitabili_30plus_per_100k_pop_totale',
             'Infermieri per 100k residenti',
             'Infermieri per 100k vs Mortalita evitabile 30+ (Euro-2013) — Regioni 2022', df)
plt.tight_layout()
plt.show()

## Scatter — Personale ospedaliero vs mortalità evitabile 30+

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(ax, 'personale_osp_per_100k', 'decessi_evitabili_30plus_per_100k_pop_totale',
             'Personale ospedaliero totale per 100k residenti',
             'Pers. ospedaliero per 100k vs Mortalita evitabile 30+ (Euro-2013) — Regioni 2022', df)
plt.tight_layout()
plt.show()

## Scatter — Posti letto vs mortalità evitabile 30+

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter_plot(ax, 'posti_letto_previsti_per_100k', 'decessi_evitabili_30plus_per_100k_pop_totale',
             'Posti letto previsti per 100k residenti',
             'Posti letto per 100k vs Mortalita evitabile 30+ (Euro-2013) — Regioni 2022', df)
plt.tight_layout()
plt.show()

## Heatmap correlazioni Pearson

In [ ]:
corr_cols = [
    'medici_mmg_per_100k',
    'infermieri_per_100k',
    'personale_osp_per_100k',
    'posti_letto_previsti_per_100k',
    'decessi_evitabili_30plus_per_100k_pop_totale',
]
corr = df[corr_cols].corr()

labels = ['MMG/100k', 'Infermieri/100k', 'Pers.osp/100k', 'PL/100k', 'Evitabile 30+/100k']

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha='right')
ax.set_yticklabels(labels)
ax.set_title('Heatmap correlazioni Pearson — indicatori chiave 2022 (Euro-2013 proxy)', fontweight='bold')
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8, color='black')
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Correlazione r')
plt.tight_layout()
plt.show()
corr

## Risposta sintetica alla domanda analitica

*Aggiornare con i valori r effettivi dopo l'esecuzione.*

### Cosa guardare rispetto a v1

- Se le correlazioni cambiano segno (da positive a negative) rispetto a v1: il confondente demografico è ridotto dal filtro per cause evitabili.
- Se restano positive: la struttura demografica domina ancora, serve age-standardizzazione esplicita per v3.
- Outlier da verificare: Molise e Valle d'Aosta su `personale_osp_per_100k` (segnalati da Gabrymi93 in PR #15 — possibili problemi di copertura dato fonte C).

*Nota metodologica: Euro-2013 proxy su 12 cause amenable/preventable, tasso grezzo 30+ (non age-standardized). Denominatore = popolazione totale regionale da fonte A. Gap temporale dati Ministero fermi al 2022.*